In [3]:
# English Model trained against ASVSpoof 2019 dataset

In [2]:
# 0. KERNEL CLEANER & PERFECT LZMA MOCK (Must be the absolute first lines)
import sys
import types

# Clear crashed modules from memory
for k in list(sys.modules.keys()):
    if k.startswith("datasets") or k.startswith("fsspec") or k.startswith("sklearn") or k == "lzma" or k == "_lzma":
        del sys.modules[k]

# Create a PERFECT Fake LZMA module to bypass the missing DLL
fake_lzma = types.ModuleType("lzma")
fake_lzma.LZMAError = Exception

class FakeLZMAFile:
    def __init__(self, *args, **kwargs): pass
    def read(self, *args, **kwargs): return b""
    def write(self, *args, **kwargs): return 0
    def seek(self, *args, **kwargs): return 0
    def tell(self, *args, **kwargs): return 0
    def close(self): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass

fake_lzma.LZMAFile = FakeLZMAFile
fake_lzma.open = lambda *args, **kwargs: FakeLZMAFile()
fake_lzma.is_check_supported = lambda check: False
fake_lzma.FORMAT_AUTO = 0
fake_lzma.FORMAT_XZ = 1
fake_lzma.FORMAT_ALONE = 2
fake_lzma.FORMAT_RAW = 3
fake_lzma.CHECK_NONE = 0
fake_lzma.CHECK_CRC32 = 1
fake_lzma.CHECK_CRC64 = 4
fake_lzma.CHECK_SHA256 = 10

# Inject the mock into system memory
sys.modules["lzma"] = fake_lzma
sys.modules["_lzma"] = fake_lzma
sys.modules["torchvision"] = None  # Block torchvision completely

# ==============================================================================
# 1. INITIALIZATION & SETUP
# ==============================================================================
import os
import torch
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

BASE_DIR = r"E:\SIH2026\models"
MODEL_DIR = os.path.join(BASE_DIR, "trained_english")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
print(f"Evaluating Model: {MODEL_DIR}")

extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_DIR)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

# 2. STREAMING CONFIGURATION
TOTAL_EVAL_SAMPLES = 2000    # 1000 Bonafide + 1000 Spoof (Fast & accurate calibration)
BATCH_SIZE = 16
TARGET_LENGTH = 48000       # 3 seconds @ 16kHz

print("\nConnecting to ASVspoof 2019 LA partition stream...")
stream_ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA",
    split="test",
    streaming=True,
    token=HF_TOKEN
)

ground_truth_labels = []
predicted_spoof_probs = []

batch_audios = []
batch_labels = []

bonafide_count = 0
spoof_count = 0
max_per_class = TOTAL_EVAL_SAMPLES // 2

def process_batch(audio_list, label_list):
    inputs = extractor(
        audio_list,
        sampling_rate=16000,
        padding=False,
        truncation=False,
        return_attention_mask=False,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()

    # Probability of index 1 (Spoof)
    batch_spoof_probs = probs[:, 1].tolist()

    ground_truth_labels.extend(label_list)
    predicted_spoof_probs.extend(batch_spoof_probs)

# 3. STREAM & INFERENCE
print(f"\nCollecting & evaluating {TOTAL_EVAL_SAMPLES} balanced samples...")

try:
    for item in stream_ds:
        raw_audio = np.array(item["audio"]["array"], dtype=np.float32)
        true_label = int(item["label"])  # 0: Bonafide, 1: Spoof

        # Maintain 50:50 balance
        if true_label == 0:
            if bonafide_count >= max_per_class:
                continue
            bonafide_count += 1
        else:
            if spoof_count >= max_per_class:
                continue
            spoof_count += 1

        # Fixed 3-second center-cropping
        if len(raw_audio) > TARGET_LENGTH:
            start_idx = (len(raw_audio) - TARGET_LENGTH) // 2
            cropped = raw_audio[start_idx : start_idx + TARGET_LENGTH]
        else:
            pad_len = TARGET_LENGTH - len(raw_audio)
            cropped = np.pad(raw_audio, (0, pad_len), 'constant')

        batch_audios.append(cropped)
        batch_labels.append(true_label)

        if len(batch_audios) == BATCH_SIZE:
            process_batch(batch_audios, batch_labels)
            batch_audios = []
            batch_labels = []
            print(f" -> Evaluated: {len(ground_truth_labels)}/{TOTAL_EVAL_SAMPLES} | Bonafide: {bonafide_count} | Spoof: {spoof_count}", end="\r")

        if len(ground_truth_labels) >= TOTAL_EVAL_SAMPLES:
            break

    # Process remaining audio buffer
    if len(batch_audios) > 0:
        process_batch(batch_audios, batch_labels)

except KeyboardInterrupt:
    print("\nProcess stopped manually. Running calibration on collected data...")

# 4. DECISION BOUNDARY CALIBRATION TABLE
y_true = np.array(ground_truth_labels)
y_scores = np.array(predicted_spoof_probs)

print("\n\n" + "="*80)
print("             EQUAL ERROR RATE (EER) & THRESHOLD CALIBRATION TABLE            ")
print("="*80)
print(f"{'Threshold':<10} | {'Bonafide Acc (Real)':<22} | {'Spoof Acc (Fake)':<20} | {'Overall Acc':<12} | {'EER Gap':<10}")
print("-" * 80)

best_threshold = 0.50
min_eer_gap = 100.0
calibration_data = []

# Sweep thresholds from 0.40 to 0.98
for th in np.arange(0.40, 0.99, 0.03):
    preds = (y_scores >= th).astype(int)

    bf_mask = (y_true == 0)
    sp_mask = (y_true == 1)

    bf_acc = (preds[bf_mask] == 0).mean() * 100
    sp_acc = (preds[sp_mask] == 1).mean() * 100
    total_acc = (preds == y_true).mean() * 100
    eer_gap = abs(bf_acc - sp_acc)

    if eer_gap < min_eer_gap:
        min_eer_gap = eer_gap
        best_threshold = th

    calibration_data.append({
        "Threshold": round(th, 2),
        "Bonafide_Acc": round(bf_acc, 2),
        "Spoof_Acc": round(sp_acc, 2),
        "Overall_Acc": round(total_acc, 2),
        "Gap": round(eer_gap, 2)
    })

    print(f"{th:<10.2f} | {bf_acc:<20.2f}% | {sp_acc:<18.2f}% | {total_acc:<10.2f}% | {eer_gap:<8.2f}%")

print("="*80)
print(f"\nRECOMMENDED CALIBRATED THRESHOLD: {best_threshold:.2f}")

opt_bf = [row['Bonafide_Acc'] for row in calibration_data if row['Threshold'] == round(best_threshold, 2)][0]
opt_sp = [row['Spoof_Acc'] for row in calibration_data if row['Threshold'] == round(best_threshold, 2)][0]
opt_total = [row['Overall_Acc'] for row in calibration_data if row['Threshold'] == round(best_threshold, 2)][0]

print(f"Expected Balanced Bonafide Accuracy : {opt_bf}%")
print(f"Expected Balanced Spoof Accuracy    : {opt_sp}%")
print(f"Expected Balanced Overall Accuracy  : {opt_total}%")
print(f"\nAction: Use this {best_threshold:.2f} value as SPOOF_THRESHOLD inside server.py.")

Device: CUDA
Evaluating Model: E:\SIH2026\models\trained_english


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]


Connecting to ASVspoof 2019 LA partition stream...

 -> Evaluated: 2000/2000 | Bonafide: 1000 | Spoof: 1000

             EQUAL ERROR RATE (EER) & THRESHOLD CALIBRATION TABLE            
Threshold  | Bonafide Acc (Real)    | Spoof Acc (Fake)     | Overall Acc  | EER Gap   
--------------------------------------------------------------------------------
0.40       | 31.50               % | 97.60             % | 64.55     % | 66.10   %
0.43       | 33.40               % | 97.30             % | 65.35     % | 63.90   %
0.46       | 34.50               % | 97.10             % | 65.80     % | 62.60   %
0.49       | 36.60               % | 96.80             % | 66.70     % | 60.20   %
0.52       | 38.00               % | 96.70             % | 67.35     % | 58.70   %
0.55       | 38.80               % | 96.50             % | 67.65     % | 57.70   %
0.58       | 40.70               % | 95.80             % | 68.25     % | 55.10   %
0.61       | 42.60               % | 95.50             % | 69.0